In [ ]:
import os
import numpy as np
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/STGCN_Data'
print(folder_path)

In [ ]:
# ============================================================
# Step 1: Load and inspect dataset
# ============================================================
DATA_DIR = folder_path  # '/content/drive/MyDrive/STGCN_Data'

# --- Auto-detect which xlsx file is the "master" (has the named sheets) ---
import openpyxl
REQUIRED_SHEETS = {'Links_by_Name', 'Links_by_from', 'Links_by_to', 'Info'}
xlsx_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.xlsx')]
master_file = None
for fname in xlsx_files:
    try:
        wb = openpyxl.load_workbook(f"{DATA_DIR}/{fname}", read_only=True, data_only=True)
        if REQUIRED_SHEETS.issubset(set(wb.sheetnames)):
            master_file = fname
            print(f"[auto] master_xlsx = {fname}  (sheets: {wb.sheetnames})")
            wb.close()
            break
        wb.close()
    except Exception as e:
        print(f"[skip] {fname}: {e}")

assert master_file is not None, f"No master xlsx found. Inspected: {xlsx_files}"

# Remaining edge files = the other xlsx files (excluding master)
other_xlsx = [f for f in xlsx_files if f != master_file]
print(f"[auto] other xlsx files: {other_xlsx}")

# Adjacency csv
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print(f"[auto] csv files: {csv_files}")

FILES = {
    'master_xlsx': master_file,
    'adj_matrix' : csv_files[0] if csv_files else 'adj03_links_adj.csv',
    'edge_fwd'   : other_xlsx[0] if len(other_xlsx) > 0 else None,
    'edge_bwd'   : other_xlsx[1] if len(other_xlsx) > 1 else None,
}
print(f"[auto] FILES = {FILES}")


def load_all():
    """Load all 8 files and return them as a dictionary."""
    d = {}
    # Master features
    d['lbn']  = pd.read_excel(f"{DATA_DIR}/{FILES['master_xlsx']}", sheet_name='Links_by_Name')
    d['lbf']  = pd.read_excel(f"{DATA_DIR}/{FILES['master_xlsx']}", sheet_name='Links_by_from')
    d['lbt']  = pd.read_excel(f"{DATA_DIR}/{FILES['master_xlsx']}", sheet_name='Links_by_to')
    d['info'] = pd.read_excel(f"{DATA_DIR}/{FILES['master_xlsx']}", sheet_name='Info', header=None)

    # Graph structure - keep only 170x170 (drop the trailing zero column)
    adj_raw = pd.read_csv(f"{DATA_DIR}/{FILES['adj_matrix']}", header=None).values
    d['adj'] = adj_raw[:170, :170].astype(np.float32)

    # Edge lists
    d['edge_fwd'] = pd.read_excel(f"{DATA_DIR}/{FILES['edge_fwd']}", header=None)
    d['edge_bwd'] = pd.read_excel(f"{DATA_DIR}/{FILES['edge_bwd']}", header=None)

    return d


def inspect(d):
    """Print a clean summary of what's in each file."""
    print("=" * 70)
    print("DATASET INSPECTION REPORT")
    print("=" * 70)

    # Master sheet
    print(f"\n[1] Links_by_Name ({d['lbn'].shape[0]} links x {d['lbn'].shape[1]} cols)")
    useful_cols = ['Name', 'Length', '#appeance', 'from', 'to',
                   'move1', 'move2', 'move3',
                   'lon_from', 'lat_from', 'lon_to', 'lat_to',
                   'haversine', '#freq']
    available = [c for c in useful_cols if c in d['lbn'].columns]
    print(f"    Available usable columns: {available}")
    print(f"    All columns: {list(d['lbn'].columns)}")
    if available:
        print(f"    Sample row: {d['lbn'].iloc[0][available].tolist()}")

    # Feature statistics
    print(f"\n[2] Key feature statistics:")
    for col in ['Length', '#appeance', '#freq']:
        if col in d['lbn'].columns:
            s = d['lbn'][col]
            print(f"    {col:12s}  min={s.min():10.3f}  max={s.max():10.3f}  "
                  f"mean={s.mean():10.3f}  NaN={s.isna().sum()}/{len(s)}")

    # Graph
    print(f"\n[3] Adjacency matrix: {d['adj'].shape}")
    print(f"    Non-zero entries: {int((d['adj'] != 0).sum())}")
    print(f"    Sparsity: {(1 - (d['adj'] != 0).mean()) * 100:.2f}%")
    print(f"    Symmetric? {np.array_equal(d['adj'], d['adj'].T)}")

    # Nodes
    if 'from' in d['lbn'].columns and 'to' in d['lbn'].columns:
        nodes = set(d['lbn']['from']) | set(d['lbn']['to'])
        print(f"\n[4] Network: {len(nodes)} unique sensor nodes, "
              f"IDs {int(min(nodes))}-{int(max(nodes))}")

    # Source files
    n_files = len(d['info'])
    print(f"\n[5] Source CSV files: {n_files} monthly files (Apr 2019 - Jan 2021)")

    print("\n" + "=" * 70)
    print("LOAD SUCCESSFUL. Proceed to Step 2.")
    print("=" * 70)


if __name__ == '__main__':
    data = load_all()
    inspect(data)


In [ ]:
# ============================================================
# STGCN for Traffic Prediction - Step 2: Preprocessing
# ============================================================
"""
Builds the spatio-temporal inputs for STGCN from the raw dataset files.

Since raw BT CSV files are NOT uploaded, we synthesise a realistic traffic
time series from #freq + a daily traffic pattern (morning/evening peaks).
This is a valid proxy for training while you wait for the real CSV data.

Outputs saved to OUT_DIR:
    sequences.npz   - X_train, y_train, X_val, y_val, X_test, y_test
    laplacian.npy   - the Chebyshev-scaled graph Laplacian
    scaler.pkl      - the fitted StandardScaler (for inverse_transform)
"""

import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
# Reuse folder_path mounted from Step 0 and master_file auto-detected in Step 1
DATA_DIR    = folder_path                       # '/content/drive/MyDrive/STGCN_Data'
OUT_DIR     = f"{folder_path}/processed"        # save artefacts in Drive (persistent)
os.makedirs(OUT_DIR, exist_ok=True)

N_LINKS     = 170          # model uses 170 of the 267 links
T_IN        = 12           # 12 x 15-min = 3 hours of history
T_OUT       = 3            # predict next 3 steps (45 minutes ahead)
TIME_STEPS  = 28800        # 300 days x 96 slots/day (simulated)
TRAIN_FRAC  = 0.70         # 70% train, 10% val, 20% test (chronological)
VAL_FRAC    = 0.10


def build_traffic_matrix(link_freq, n_steps=TIME_STEPS, seed=42):
    """Synthesise a plausible travel-time matrix of shape (T, N)."""
    np.random.seed(seed)
    N = len(link_freq)
    T = n_steps

    # Base travel time: higher #freq = busier road -> slower baseline speed
    congestion = 0.2 + link_freq * 0.8

    # Time-of-day pattern: 96 slots per 24h day
    t_idx   = np.arange(T) % 96
    day_idx = (np.arange(T) // 96) % 7

    # Morning peak ~slot 34 (08:30), evening peak ~slot 70 (17:30)
    morning = np.exp(-((t_idx - 34) ** 2) / 40.0)
    evening = np.exp(-((t_idx - 70) ** 2) / 40.0)
    daily_pattern = 0.5 + morning + evening

    # Weekends (day 5, 6) are 40% quieter
    weekend_mask = np.where(day_idx >= 5, 0.6, 1.0)
    daily_pattern = daily_pattern * weekend_mask

    # Build matrix shape (T, N) - outer product of patterns
    travel_time = np.outer(daily_pattern, 30 * congestion)
    travel_time += np.random.normal(0, 2, size=(T, N))
    travel_time = np.clip(travel_time, 5, None)

    return travel_time.astype(np.float32)


def scaled_laplacian(adj):
    """Compute Chebyshev-scaled normalised Laplacian: L_tilde = (2/lmax)*L_norm - I."""
    # Symmetrise - STGCN requires undirected graph for Chebyshev polynomials
    adj = np.maximum(adj, adj.T)

    d = adj.sum(axis=1)
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
    D_inv_sqrt = np.diag(d_inv_sqrt)

    L = np.eye(adj.shape[0]) - D_inv_sqrt @ adj @ D_inv_sqrt
    lambda_max = np.linalg.eigvalsh(L).max()
    L_tilde = (2.0 / lambda_max) * L - np.eye(adj.shape[0])
    return L_tilde.astype(np.float32)


def make_sequences(data, t_in, t_out):
    """Sliding window: X (samples, t_in, N, 1), Y (samples, t_out, N)."""
    T, N = data.shape
    n_samples = T - t_in - t_out + 1
    X = np.zeros((n_samples, t_in,  N, 1), dtype=np.float32)
    Y = np.zeros((n_samples, t_out, N),     dtype=np.float32)
    for i in range(n_samples):
        X[i, :, :, 0] = data[i : i + t_in]
        Y[i]          = data[i + t_in : i + t_in + t_out]
    return X, Y


def main():
    print("=" * 70)
    print("STEP 2: PREPROCESSING")
    print("=" * 70)

    # 1. Load link metadata and pick the first N_LINKS links
    print("\n[1/6] Loading link metadata...")
    lbn = pd.read_excel(f"{DATA_DIR}/{master_file}", sheet_name='Links_by_Name')
    link_names = lbn['Name'].values[:N_LINKS]
    link_freq  = lbn['#freq'].values[:N_LINKS].astype(np.float32)
    print(f"    -> Using {N_LINKS} links, #freq range = "
          f"[{link_freq.min():.3f}, {link_freq.max():.3f}]")

    # 2. Load adjacency matrix (auto-detect the csv)
    print("\n[2/6] Loading adjacency matrix...")
    csv_files_local = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    adj_csv = csv_files_local[0]
    adj = pd.read_csv(f"{DATA_DIR}/{adj_csv}",
                      header=None).values[:N_LINKS, :N_LINKS].astype(np.float32)
    print(f"    -> File: {adj_csv}, Shape: {adj.shape}, non-zero: {int((adj != 0).sum())}")

    # 3. Build synthetic traffic matrix
    print(f"\n[3/6] Building synthetic traffic matrix ({TIME_STEPS} steps x {N_LINKS} links)...")
    traffic = build_traffic_matrix(link_freq, n_steps=TIME_STEPS)
    print(f"    -> Shape: {traffic.shape}, "
          f"range = [{traffic.min():.2f}, {traffic.max():.2f}] sec")

    # 4. Normalise - z-score, fit on train split only to prevent data leakage
    print("\n[4/6] Normalising (z-score on train split only)...")
    n_train = int(TRAIN_FRAC * traffic.shape[0])
    scaler = StandardScaler()
    scaler.fit(traffic[:n_train])
    traffic_scaled = scaler.transform(traffic)
    print(f"    -> mean={scaler.mean_.mean():.3f}, std={scaler.scale_.mean():.3f}")

    # 5. Compute graph Laplacian
    print("\n[5/6] Computing Chebyshev-scaled Laplacian...")
    L_tilde = scaled_laplacian(adj)
    print(f"    -> Shape: {L_tilde.shape}, "
          f"eigenvalue range = [{L_tilde.min():.3f}, {L_tilde.max():.3f}]")

    # 6. Create sliding-window sequences and split chronologically
    print(f"\n[6/6] Creating sequences (T_in={T_IN}, T_out={T_OUT})...")
    X, Y = make_sequences(traffic_scaled, T_IN, T_OUT)
    print(f"    -> X shape: {X.shape}, Y shape: {Y.shape}")

    n = X.shape[0]
    n_tr = int(TRAIN_FRAC * n)
    n_va = int(VAL_FRAC * n)
    X_train, y_train = X[:n_tr],              Y[:n_tr]
    X_val,   y_val   = X[n_tr:n_tr + n_va],   Y[n_tr:n_tr + n_va]
    X_test,  y_test  = X[n_tr + n_va:],       Y[n_tr + n_va:]
    print(f"    -> Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

    # Save everything
    np.savez_compressed(f'{OUT_DIR}/sequences.npz',
                        X_train=X_train, y_train=y_train,
                        X_val=X_val,     y_val=y_val,
                        X_test=X_test,   y_test=y_test)
    np.save(f'{OUT_DIR}/laplacian.npy', L_tilde)
    with open(f'{OUT_DIR}/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    print("\n" + "=" * 70)
    print(f"[OK] Saved sequences.npz, laplacian.npy, scaler.pkl in {OUT_DIR}/")
    print("=" * 70)
    return X_train, y_train, X_val, y_val, X_test, y_test, L_tilde, scaler


if __name__ == '__main__':
    X_train, y_train, X_val, y_val, X_test, y_test, L_tilde, scaler = main()


In [ ]:
# ============================================================
# STGCN for Traffic Prediction - Step 3: Model Architecture
# ============================================================
"""
Implements the Spatio-Temporal Graph Convolutional Network from
Yu et al., IJCAI 2018 - in pure Keras / TensorFlow.

Architecture (sandwich design):
    Input: (batch, T_in=12, N=170, 1)
       -> ST-Block 1: [TemporalGatedConv -> ChebGraphConv -> TemporalGatedConv]
       -> ST-Block 2: [TemporalGatedConv -> ChebGraphConv -> TemporalGatedConv]
       -> OutputLayer: TemporalConv -> reshape
    Output: (batch, T_out=3, N=170)
"""

import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np


# ------------------------------------------------------------
# 1. CHEBYSHEV GRAPH CONVOLUTION - the "spatial" part
# ------------------------------------------------------------
class ChebGraphConv(layers.Layer):
    """
    Graph convolution using Chebyshev polynomials of the Laplacian.
    Y = sum_{k=0..K-1}  T_k(L_tilde) @ X @ W_k
    where T_0 = I, T_1 = L_tilde, T_k = 2*L_tilde*T_{k-1} - T_{k-2}
    """

    def __init__(self, out_channels, K=3, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.K = K

    def build(self, input_shape):
        in_channels = input_shape[-1]
        self.W = self.add_weight(
            name='cheb_W',
            shape=(self.K, in_channels, self.out_channels),
            initializer='glorot_uniform',
            trainable=True,
        )
        self.b = self.add_weight(
            name='cheb_b',
            shape=(self.out_channels,),
            initializer='zeros',
            trainable=True,
        )

    def call(self, inputs, L_tilde):
        x = inputs
        T0 = x
        out = tf.einsum('btnc,cf->btnf', T0, self.W[0])
        if self.K > 1:
            T1 = tf.einsum('nm,btmc->btnc', L_tilde, x)
            out += tf.einsum('btnc,cf->btnf', T1, self.W[1])
            for k in range(2, self.K):
                T2 = 2 * tf.einsum('nm,btmc->btnc', L_tilde, T1) - T0
                out += tf.einsum('btnc,cf->btnf', T2, self.W[k])
                T0, T1 = T1, T2
        return out + self.b


# ------------------------------------------------------------
# 2. GATED TEMPORAL CONVOLUTION - the "temporal" part (GLU)
# ------------------------------------------------------------
class TemporalGatedConv(layers.Layer):
    """
    1-D convolution along the time axis, with Gated Linear Unit (GLU).
    Output time dimension:  T_out = T_in - K_t + 1
    """

    def __init__(self, out_channels, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            filters=2 * self.out_channels,
            kernel_size=(self.kernel_size, 1),
            padding='valid',
            use_bias=True,
            name='gated_conv',
        )

    def call(self, inputs):
        x = self.conv(inputs)                              # (B, T', N, 2F)
        a, b = tf.split(x, num_or_size_splits=2, axis=-1)  # each (B, T', N, F)
        return a * tf.sigmoid(b)                           # GLU


# ------------------------------------------------------------
# 3. ST-BLOCK - temporal -> spatial -> temporal sandwich
# ------------------------------------------------------------
class STBlock(layers.Layer):
    """One ST-Block: Temporal -> Spatial -> Temporal, with LayerNorm + dropout."""

    def __init__(self, temporal_filters, spatial_filters,
                 kernel_size=3, K=3, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.tc1 = TemporalGatedConv(temporal_filters, kernel_size)
        self.gc  = ChebGraphConv(spatial_filters, K=K)
        self.tc2 = TemporalGatedConv(temporal_filters, kernel_size)
        self.ln  = layers.LayerNormalization()
        self.drop = layers.Dropout(dropout)

    def call(self, x, L_tilde, training=False):
        x = self.tc1(x)
        x = tf.nn.relu(self.gc(x, L_tilde))
        x = self.tc2(x)
        x = self.ln(x)
        return self.drop(x, training=training)


# ------------------------------------------------------------
# 4. FULL STGCN MODEL
# ------------------------------------------------------------
class STGCN(Model):
    """
    Full STGCN network.
    Input  : (batch, T_in=12, N=170, 1)
    Output : (batch, T_out=3, N=170)
    """

    def __init__(self, num_nodes=170, t_in=12, t_out=3,
                 block_filters=(16, 32),
                 kernel_size=3, K=3, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_nodes = num_nodes
        self.t_in = t_in
        self.t_out = t_out

        f1, f2 = block_filters
        self.block1 = STBlock(f1, f1, kernel_size, K, dropout, name='st_block_1')
        self.block2 = STBlock(f2, f2, kernel_size, K, dropout, name='st_block_2')

        # After 2 blocks of 2 temporal convs (kernel=3):
        #   T = 12 - 2*(3-1) - 2*(3-1) = 4
        self.out_temporal = layers.Conv2D(
            filters=t_out,
            kernel_size=(4, 1),
            padding='valid',
            name='output_conv',
        )
        self.L_tilde = None

    def set_laplacian(self, L_tilde):
        """Register L_tilde as a non-trainable tensor."""
        self.L_tilde = tf.constant(L_tilde, dtype=tf.float32)

    def call(self, inputs, training=False):
        assert self.L_tilde is not None, "call set_laplacian(L_tilde) before fit()"
        x = self.block1(inputs, self.L_tilde, training=training)
        x = self.block2(x,      self.L_tilde, training=training)
        x = self.out_temporal(x)              # (B, 1, N, t_out)
        x = tf.squeeze(x, axis=1)             # (B, N, t_out)
        x = tf.transpose(x, [0, 2, 1])        # (B, t_out, N)
        return x


# ------------------------------------------------------------
# 5. BUILD AND SUMMARY
# ------------------------------------------------------------
def build_model(num_nodes=170, t_in=12, t_out=3, lr=1e-3):
    model = STGCN(num_nodes=num_nodes, t_in=t_in, t_out=t_out)
    model.build(input_shape=(None, t_in, num_nodes, 1))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae'],
    )
    return model


if __name__ == '__main__':
    print("Building STGCN...")
    model = build_model()

    # Use the real Laplacian we computed in Step 2 if available, else identity
    try:
        model.set_laplacian(L_tilde)
        print("    -> using L_tilde from Step 2")
    except NameError:
        fake_L = np.eye(170, dtype=np.float32)
        model.set_laplacian(fake_L)
        print("    -> L_tilde from Step 2 not in scope; using identity")

    # Forward pass to confirm shapes
    dummy = np.random.randn(2, 12, 170, 1).astype(np.float32)
    out = model(dummy, training=False)
    print(f"\nInput shape : {dummy.shape}")
    print(f"Output shape: {out.shape}   <- should be (2, 3, 170)")

    n_params = sum(tf.size(v).numpy() for v in model.trainable_variables)
    print(f"\nTotal trainable parameters: {n_params:,}")
    print("\n[OK] Model builds correctly. Proceed to Step 4 (training).")


In [ ]:
# ============================================================
# Step 4 (preview): Exploratory & Diagnostic Plots
# ============================================================
"""
Five visualisations to sanity-check the data and graph that
will feed the STGCN model:

    1. Adjacency-matrix heatmap (graph structure)
    2. Synthetic traffic time-series (daily / weekly pattern)
    3. Feature distributions (Length, #freq, #appeance)
    4. Spatial network from lon/lat (the road map)
    5. Laplacian eigenvalue spectrum
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

# Reload pieces we need (they exist in scope from earlier steps,
# but re-loading keeps this cell self-contained if re-run)
lbn_full = pd.read_excel(f"{DATA_DIR}/{master_file}", sheet_name='Links_by_Name')
adj_csv_local = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')][0]
adj_full = pd.read_csv(f"{DATA_DIR}/{adj_csv_local}", header=None).values
adj170   = adj_full[:N_LINKS, :N_LINKS].astype(np.float32)

# --------------------------------------------------------
# 1) Adjacency matrix heatmap
# --------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

im0 = ax[0].imshow(adj170, cmap='viridis', aspect='auto')
ax[0].set_title(f"Adjacency matrix  (170x170, non-zero = {int((adj170 != 0).sum())})")
ax[0].set_xlabel("link j"); ax[0].set_ylabel("link i")
plt.colorbar(im0, ax=ax[0], fraction=0.046)

# Symmetrised version (what STGCN actually uses)
adj_sym = np.maximum(adj170, adj170.T)
im1 = ax[1].imshow(adj_sym, cmap='magma', aspect='auto')
ax[1].set_title(f"Symmetrised adjacency  (non-zero = {int((adj_sym != 0).sum())})")
ax[1].set_xlabel("link j"); ax[1].set_ylabel("link i")
plt.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

# --------------------------------------------------------
# 2) Synthetic traffic time series for a few sample links
# --------------------------------------------------------
# Re-build a small slice of the traffic so we plot on raw seconds
link_freq = lbn_full['#freq'].values[:N_LINKS].astype(np.float32)
traffic_demo = build_traffic_matrix(link_freq, n_steps=96 * 14)   # 14 days

# Choose 4 links spanning the #freq range (low, mid-low, mid-high, high)
order = np.argsort(link_freq)
sample_idx = [order[5], order[N_LINKS // 3], order[2 * N_LINKS // 3], order[-5]]
labels = [f"link #{i} (freq={link_freq[i]:.2f})" for i in sample_idx]

t_hours = np.arange(traffic_demo.shape[0]) / 4.0   # 96 slots/day -> 4 slots/hour
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

for i, lab in zip(sample_idx, labels):
    ax[0].plot(t_hours, traffic_demo[:, i], lw=0.9, label=lab)
ax[0].set_title("Synthetic travel time over 14 days  (4 sample links)")
ax[0].set_ylabel("travel time (sec)")
ax[0].legend(loc='upper right', fontsize=8)

# Average daily profile (averaged over all 170 links, all days)
T = traffic_demo.shape[0]
avg_day = traffic_demo.reshape(T // 96, 96, N_LINKS).mean(axis=(0, 2))
ax[1].plot(np.arange(96) / 4.0, avg_day, color='crimson', lw=2)
ax[1].set_title("Average daily profile (mean across all links & days)")
ax[1].set_xlabel("hour of day")
ax[1].set_ylabel("avg travel time (sec)")
ax[1].axvline(8.5,  ls='--', color='gray', alpha=0.6, label='morning peak')
ax[1].axvline(17.5, ls='--', color='gray', alpha=0.6, label='evening peak')
ax[1].legend()
plt.tight_layout(); plt.show()

# --------------------------------------------------------
# 3) Feature distributions
# --------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

for j, col in enumerate(['Length', '#freq', '#appeance']):
    if col in lbn_full.columns:
        s = lbn_full[col].dropna()
        ax[j].hist(s, bins=40, color='steelblue', edgecolor='white')
        ax[j].set_title(f"{col}  (n={len(s)})")
        ax[j].set_xlabel(col); ax[j].set_ylabel('count')
        if col == '#appeance':
            ax[j].set_yscale('log')
plt.tight_layout(); plt.show()

# --------------------------------------------------------
# 4) Spatial network (lon/lat) with adjacency edges
# --------------------------------------------------------
have_coords = all(c in lbn_full.columns for c in
                  ['lon_from', 'lat_from', 'lon_to', 'lat_to'])

if have_coords:
    sub = lbn_full.iloc[:N_LINKS]
    fig, ax = plt.subplots(figsize=(9, 8))

    # Plot every link as a thin line from (lon_from, lat_from) to (lon_to, lat_to)
    for _, row in sub.iterrows():
        ax.plot([row['lon_from'], row['lon_to']],
                [row['lat_from'], row['lat_to']],
                color='lightgray', lw=0.6, zorder=1)

    # Compute a simple "node coordinate" = midpoint of each link
    mids = np.column_stack([
        (sub['lon_from'].values + sub['lon_to'].values) / 2.0,
        (sub['lat_from'].values + sub['lat_to'].values) / 2.0,
    ])

    # Draw adjacency edges between link-midpoints
    ii, jj = np.where(adj_sym > 0)
    mask = ii < jj                       # avoid duplicates
    for i, j in zip(ii[mask], jj[mask]):
        ax.plot([mids[i, 0], mids[j, 0]],
                [mids[i, 1], mids[j, 1]],
                color='steelblue', alpha=0.35, lw=0.5, zorder=2)

    # Colour each midpoint by #freq
    sc = ax.scatter(mids[:, 0], mids[:, 1],
                    c=link_freq, cmap='plasma', s=22, zorder=3,
                    edgecolor='black', linewidth=0.3)
    plt.colorbar(sc, ax=ax, label='#freq')
    ax.set_title("Sensor / link spatial layout  (edges = adjacency)")
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    ax.set_aspect('equal', 'box')
    plt.tight_layout(); plt.show()
else:
    print("[skip] lon/lat columns not in master sheet")

# --------------------------------------------------------
# 5) Eigenvalue spectrum of the Chebyshev-scaled Laplacian
# --------------------------------------------------------
# Recompute eigenvalues of the un-scaled normalised Laplacian L
d = adj_sym.sum(axis=1)
d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
L_norm = np.eye(N_LINKS) - (d_inv_sqrt[:, None] * adj_sym * d_inv_sqrt[None, :])
eig_norm = np.linalg.eigvalsh(L_norm)
eig_tilde = np.linalg.eigvalsh(L_tilde)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(eig_norm, 'o-', ms=2, color='teal')
ax[0].set_title(f"Normalised Laplacian eigenvalues  (lambda_max={eig_norm.max():.3f})")
ax[0].set_xlabel("index"); ax[0].set_ylabel("lambda")

ax[1].plot(eig_tilde, 'o-', ms=2, color='darkorange')
ax[1].set_title("Chebyshev-scaled Laplacian L_tilde eigenvalues  (in [-1, 1])")
ax[1].set_xlabel("index"); ax[1].set_ylabel("lambda")
ax[1].axhline(1.0,  ls='--', color='gray', alpha=0.5)
ax[1].axhline(-1.0, ls='--', color='gray', alpha=0.5)
plt.tight_layout(); plt.show()

print("\n[OK] Plots rendered.")


In [ ]:
# ============================================================
# STGCN for Traffic Prediction - Step 5: Training & Evaluation
# ============================================================
"""
Trains the STGCN from Step 3 on the sequences from Step 2, then reports
MAE / RMSE / MAPE in REAL UNITS (seconds) per forecast horizon.

Self-contained: reloads sequences.npz / laplacian.npy / scaler.pkl from
OUT_DIR, so it can be re-run without re-running Steps 1-2.

Artefacts written to OUT_DIR:
    stgcn_best.weights.h5   - best weights (lowest val loss)
    history.csv             - per-epoch loss/mae
    metrics.csv             - per-horizon MAE/RMSE/MAPE on the test set
"""

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
OUT_DIR    = f"{folder_path}/processed"
EPOCHS     = 60
BATCH_SIZE = 64
PATIENCE   = 10
SEED       = 42

tf.keras.utils.set_random_seed(SEED)

print("=" * 70)
print("STEP 5: TRAINING")
print("=" * 70)

# ---------------------------------------------------------
# 1. Reload artefacts
# ---------------------------------------------------------
print("\n[1/5] Loading artefacts...")
z = np.load(f"{OUT_DIR}/sequences.npz")
X_train, y_train = z['X_train'], z['y_train']
X_val,   y_val   = z['X_val'],   z['y_val']
X_test,  y_test  = z['X_test'],  z['y_test']
L_tilde = np.load(f"{OUT_DIR}/laplacian.npy")
with open(f"{OUT_DIR}/scaler.pkl", 'rb') as f:
    scaler = pickle.load(f)

print(f"    -> train {X_train.shape}  val {X_val.shape}  test {X_test.shape}")
print(f"    -> L_tilde {L_tilde.shape}")

T_OUT = y_train.shape[1]
N     = y_train.shape[2]

# ---------------------------------------------------------
# 2. Build model
# ---------------------------------------------------------
print("\n[2/5] Building model...")
model = build_model(num_nodes=N, t_in=X_train.shape[1], t_out=T_OUT, lr=1e-3)
model.set_laplacian(L_tilde)

# Force a real build with a forward pass (avoids the subclassed-Model
# "unbuilt state" warning and makes checkpointing reliable)
_ = model(X_train[:2], training=False)
n_params = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"    -> trainable parameters: {n_params:,}")

# ---------------------------------------------------------
# 3. Train
# ---------------------------------------------------------
print("\n[3/5] Training...")
ckpt_path = f"{OUT_DIR}/stgcn_best.weights.h5"
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-5, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_loss',
        save_best_only=True, save_weights_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)

hist_df = pd.DataFrame(history.history)
hist_df.index.name = 'epoch'
hist_df.to_csv(f"{OUT_DIR}/history.csv")
print(f"    -> best val_loss = {min(history.history['val_loss']):.5f} "
      f"at epoch {int(np.argmin(history.history['val_loss'])) + 1}")

# ---------------------------------------------------------
# 4. Evaluate on the test set, in real units
# ---------------------------------------------------------
print("\n[4/5] Evaluating on test set...")
y_pred_scaled = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

# StandardScaler was fitted per-link on (T, N), so mean_/scale_ have shape (N,)
# and broadcast across the (samples, T_OUT, N) arrays.
def inverse(a):
    return a * scaler.scale_[None, None, :] + scaler.mean_[None, None, :]

y_pred = inverse(y_pred_scaled)
y_true = inverse(y_test)

def horizon_metrics(true, pred):
    err  = pred - true
    mae  = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    mask = np.abs(true) > 1e-3          # guard against divide-by-zero
    mape = np.mean(np.abs(err[mask] / true[mask])) * 100
    return mae, rmse, mape

rows = []
for h in range(T_OUT):
    mae, rmse, mape = horizon_metrics(y_true[:, h, :], y_pred[:, h, :])
    rows.append({'horizon': f"{(h + 1) * 15} min",
                 'MAE_sec': mae, 'RMSE_sec': rmse, 'MAPE_pct': mape})
mae, rmse, mape = horizon_metrics(y_true, y_pred)
rows.append({'horizon': 'overall',
             'MAE_sec': mae, 'RMSE_sec': rmse, 'MAPE_pct': mape})

metrics = pd.DataFrame(rows)
metrics.to_csv(f"{OUT_DIR}/metrics.csv", index=False)
print()
print(metrics.to_string(index=False,
                        float_format=lambda v: f"{v:8.4f}"))

# Naive baselines for context -- a model that beats neither is not learning.
# Persistence: predict the last observed value for every horizon.
last_obs = inverse(np.repeat(X_test[:, -1, :, 0][:, None, :], T_OUT, axis=1))
b_mae, b_rmse, b_mape = horizon_metrics(y_true, last_obs)
print(f"\n    Persistence baseline : MAE {b_mae:7.4f}  "
      f"RMSE {b_rmse:7.4f}  MAPE {b_mape:6.2f}%")
print(f"    STGCN                : MAE {mae:7.4f}  "
      f"RMSE {rmse:7.4f}  MAPE {mape:6.2f}%")
improve = (b_mae - mae) / b_mae * 100
print(f"    -> STGCN improves MAE by {improve:.1f}% over persistence")

# ---------------------------------------------------------
# 5. Diagnostic plots
# ---------------------------------------------------------
print("\n[5/5] Plotting...")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist_df['loss'], label='train')
ax[0].plot(hist_df['val_loss'], label='val')
ax[0].set_title('Huber loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist_df['mae'], label='train')
ax[1].plot(hist_df['val_mae'], label='val')
ax[1].set_title('MAE (scaled units)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

# Predicted vs actual for one link over ~2 days, at the 45-min horizon
link = int(np.argmax(y_true.std(axis=(0, 1))))   # most variable link
span = 96 * 2
h = T_OUT - 1
plt.figure(figsize=(13, 4))
plt.plot(y_true[:span, h, link], lw=1.4, label='actual')
plt.plot(y_pred[:span, h, link], lw=1.4, ls='--', label='predicted')
plt.title(f"Link #{link} - {(h + 1) * 15} min ahead (first {span} test steps)")
plt.xlabel('test time step'); plt.ylabel('travel time (sec)')
plt.legend(); plt.tight_layout(); plt.show()

# Error distribution
plt.figure(figsize=(7, 4))
plt.hist((y_pred - y_true).ravel(), bins=80,
         color='steelblue', edgecolor='white')
plt.title('Prediction error distribution (sec)')
plt.xlabel('predicted - actual'); plt.ylabel('count')
plt.tight_layout(); plt.show()

print("\n" + "=" * 70)
print(f"[OK] Saved stgcn_best.weights.h5, history.csv, metrics.csv in {OUT_DIR}/")
print("NOTE: these metrics are on SYNTHETIC data. They validate the pipeline,")
print("      not the science. Re-run once the real BT CSVs are in place.")
print("=" * 70)
